# 49 -- flip-equivariance probe, step 2: the paired GPU inference probe

**Needs a GPU pod. Cannot run on this machine** (verified: this Mac has the FRAME parquets
but not the source videos, so `FrameProvider` has nothing to decode). Written and reviewed
here; execute on the pod that has `orena-data` and the rung-42 checkpoint.

**Question, restated from the experiment README:** does rung 42 ep4 (`checkpoint-4848`,
submission 03, 0.5809 on the platform) correctly track object LOCATION when the image is
horizontally flipped at inference time? Step 1 found **175 transformable rows** on this
checkpoint's own leak-free held-out set (8 videos / 1,283 questions). This notebook answers
each of those 175 rows TWICE -- once on the untouched frame, once on the same frame
horizontally mirrored with the matching label-correct question/gold from `flip_audit.py`
-- in **one loaded-model session**, then scores both conditions through the SAME canonical
path every other rung is scored through: `focus.evaluation.Evaluator` with the real LLM
judge, not a bespoke comparator.

**Two reads, not one:**
1. **Paired accuracy delta** (flip minus original), video-clustered CI, exactly
   `frame.metrics.paired_delta_ci` -- the same instrument rung 42's own eval notebook used
   to compare itself against A2.
2. **Literal answer equivariance**, judge-free and stricter: for the two rules whose ANSWER
   carries a quadrant token (`object_center_quadrant`, `all_object_positions`), does
   `flip_audit.swap_left_right_quadrants(original_prediction)` actually equal the flipped
   prediction? A model reading pixels should move its answer with the mirror; a model
   reading a memorised prior should not. `fixed_quadrant_class` has no quadrant token in its
   ANSWER (only in the question), so this check does not apply there -- its accuracy delta
   is the read.

**Same session, not archived data**, for both conditions -- avoids the ~0.5%-of-answers
GPU-swap drift [[archived-results-not-bit-reproducible]] documents, which would otherwise be
indistinguishable from a real equivariance failure at this sample size (n=175, 8 videos).

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, sys, time
from pathlib import Path
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "49-flip-equivariance":
    EXP = REPO / "experiments" / "49-flip-equivariance"

for p in (
    REPO / "src",
    REPO / "vendor" / "orena-focus" / "src",
    REPO / "experiments" / "24-geometric-aug" / "_models",  # flip_audit.py, owned by rung 24
    EXP / "_tools",                                          # flip_pair_runner.py, this rung's own
):
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# `swift` is shelled out to (via `swift.cli.main`, not the console script -- rung 48's fix)
# for the merge. A papermill kernel does NOT inherit the env's bin/ on PATH (rung 39's scar).
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

# HF_HOME before the offline flags mean anything, and before any HF import (rung 42's scar --
# the judge lives under .cache/huggingface, NOT hf_cache, which holds the 27B/32B only).
os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(name)s %(message)s", datefmt="%H:%M:%S")

import flip_audit
from flip_pair_runner import merge_adapter, materialize_flip_pairs, answer_paired
from frame.config import BaselineConfig
from frame.data import FrameProvider, load_frame_items
from frame import metrics
from focus.data.data_models import Request, Reference, Response, save_items
from focus.evaluation.evaluator import Evaluator
from focus.evaluation.judges import TransformersJudge

print("repo:", REPO, "| exp:", EXP, "| swift on PATH:", (Path(_envbin) / "swift").exists())

In [ ]:
# --- parameters (RAW LITERALS ONLY -- papermill injects BELOW this cell) --------
SMOKE = False          # True -> 8 rows (one per rule, roughly), wiring only, no verdict
N_SMOKE = 8

RUN = "49_flip_pair_v1"
AUDIT_CSV = "experiments/49-flip-equivariance/RESULTS_flip_audit_heldout8.csv"
SPLIT_42_JSON = "experiments/42-merged-corpus/RESULTS_split_42.json"

# rung 42 ep4 -- submission 03's checkpoint, 0.5809 on the platform. Pod-specific path from
# submission 03's README (`swift export --merge_lora true` source); confirm it still exists
# before trusting this literal -- checkpoints get reclaimed (rung 42's own notebook deletes
# each merge after scoring to protect the volume's ~60 GB headroom).
ADAPTER = ("/workspace/repo_rodri/experiments/42-merged-corpus/runs/42_merged_v1/"
           "ckpt/v0-20260814-163049/checkpoint-4848")
BASE_MODEL = "/workspace/models/qwen3-vl-8b"
DATA_ROOT = "/workspace/orena-data"
DEVICE = "cuda"
SEED = 42
N_BOOT = 4000

In [ ]:
# --- derived --------------------------------------------------------------------
RUN_DIR = EXP / "runs" / RUN
MERGED_DIR = RUN_DIR / "merged" / "checkpoint-4848"
FRAMES_DIR = RUN_DIR / "frames"
RUN_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found -- pull the QA parquets"

audit = pd.read_csv(REPO / AUDIT_CSV)
audit = audit[audit["disposition"] == "transformable"].copy()
assert len(audit) == 175, f"expected 175 transformable rows from step 1, got {len(audit)}"
audit["qID"] = audit["dataset"] + "__" + audit["id"].astype(str)
if SMOKE:
    # one row per rule, roughly, so the smoke exercises all three code paths
    audit = audit.groupby("rule", group_keys=False).head(max(1, N_SMOKE // 3)).head(N_SMOKE)
print(f"transformable rows this run: {len(audit)}  (rules: {audit['rule'].value_counts().to_dict()})")

In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ------
# Evaluator loads the judge only at scoring time, well after the merge and the whole
# inference pass -- a missing cache would otherwise fail after everything expensive is
# already paid for (rung 42's own scar, same gate, same reasoning).
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. Fix the env -- do NOT disable the offline "
        "flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")

In [ ]:
# --- GATE: every audit qID resolves to exactly one FrameItem on rung 42's held-out set ---
split_42 = json.loads((REPO / SPLIT_42_JSON).read_text())
HELD = {tuple(v.split("/", 1)) for v in split_42["videos_held_out"]}

_all_items = load_frame_items(BaselineConfig(data_root=DATA_ROOT))
_held_items = [i for i in _all_items if (i.dataset, i.video_id) in HELD]
by_qid = {i.request.qID: i for i in _held_items}

missing = set(audit["qID"]) - set(by_qid)
assert not missing, f"{len(missing)} audit qIDs not found among the 8 held-out videos: {missing}"

items = [by_qid[q] for q in audit["qID"]]
# sanity: the question text on the FrameItem must match what step 1's audit recorded --
# a mismatch would mean the two are reading different data vintages.
for it, (_, row) in zip(items, audit.iterrows()):
    assert it.request.question == row["question"], (
        f"question text mismatch for {it.request.qID} -- audit CSV and live parquet disagree"
    )
items.sort(key=lambda it: (it.dataset, it.video_id, it.frame_index))  # one reader open at a time
print(f"resolved {len(items)} FrameItems, all question text verified against step 1's audit")
del _all_items

In [ ]:
# --- merge the adapter, materialize original + flipped frames -----------------------
if MERGED_DIR.is_dir() and (MERGED_DIR / "config.json").exists():
    print("already merged ->", MERGED_DIR)
else:
    t0 = time.perf_counter()
    merge_adapter(BASE_MODEL, ADAPTER, MERGED_DIR)
    print(f"merged in {time.perf_counter() - t0:.0f}s ->", MERGED_DIR)

provider = FrameProvider(BaselineConfig(data_root=DATA_ROOT))
materialize_flip_pairs(items, provider, FRAMES_DIR)
print(f"frames materialized under {FRAMES_DIR} "
      f"({2 * len(items)} files: original + flipped per row)")

In [ ]:
# --- build the paired item list: original rows + flipped rows, ONE inference call ----
audit_by_qid = audit.set_index("qID")

pair_rows = []
for it in items:
    qid = it.request.qID
    row = audit_by_qid.loc[qid]
    pair_rows.append({"qID": qid, "condition": "orig",
                      "image_path": str(FRAMES_DIR / f"{qid}.jpg"), "question": row["question"]})
    pair_rows.append({"qID": f"{qid}__flip", "condition": "flip",
                      "image_path": str(FRAMES_DIR / f"{qid}__flip.jpg"),
                      "question": row["flipped_question"]})
pair_items = pd.DataFrame(pair_rows)
assert len(pair_items) == 2 * len(items)
print(f"{len(pair_items)} items queued ({len(items)} original + {len(items)} flipped)")

In [ ]:
# --- run inference: BOTH conditions, ONE loaded model, same GPU session -------------
t0 = time.perf_counter()
preds = answer_paired(MERGED_DIR, pair_items, device=DEVICE)
print(f"inference done in {(time.perf_counter() - t0) / 60:.1f} min")

n_err = int(preds["prediction"].str.startswith("Inference Error:").sum())
if n_err:
    share = n_err / len(preds)
    print(f"G-INFER: {n_err}/{len(preds)} generations failed ({share:.1%})")
    # same threshold run_baseline uses: a broken engine must not be scored as incapacity
    assert share <= 0.01, f"G-INFER: {share:.1%} of generations failed -- fix the engine first"
preds.to_csv(RUN_DIR / "predictions_raw.csv", index=False)
print("wrote", RUN_DIR / "predictions_raw.csv")

In [ ]:
# --- build Request/Reference/Response for BOTH conditions, score through the SAME ----
# canonical Evaluator + real judge every other rung is scored through. Flip qIDs carry a
# `__flip` SUFFIX (not prefix), so `qID.split("__", 1)[0]` still reads the right dataset for
# the OOD stamp below -- the same convention `run_baseline` relies on.
preds_by_qid = preds.set_index("qID")

requests, references, responses = [], [], []
for it in items:
    qid = it.request.qID
    row = audit_by_qid.loc[qid]

    requests.append(it.request)
    references.append(it.reference)
    responses.append(Response(qID=qid, content=preds_by_qid.loc[qid, "prediction"],
                              latency=float(preds_by_qid.loc[qid, "latency"])))

    fqid = f"{qid}__flip"
    requests.append(Request(qID=fqid, videoID=it.request.videoID,
                            start_time=it.request.start_time, end_time=it.request.end_time,
                            procedure_type=it.request.procedure_type,
                            question=row["flipped_question"]))
    references.append(Reference(qID=fqid, primary=it.reference.primary,
                                _format=it.reference._format, answer=row["flipped_answer"],
                                format_kwargs=it.reference.format_kwargs,
                                secondaries=it.reference.secondaries,
                                ood=it.reference.ood, clinical=it.reference.clinical))
    responses.append(Response(qID=fqid, content=preds_by_qid.loc[fqid, "prediction"],
                              latency=float(preds_by_qid.loc[fqid, "latency"])))

for ref in references:  # same OOD stamp run_baseline applies -- see data.py:110
    ref.ood = ref.qID.split("__", 1)[0] == "heico"

save_items(responses, RUN_DIR / "responses.json")
save_items(requests, RUN_DIR / "requests.json")
save_items(references, RUN_DIR / "references.json")

judge = TransformersJudge(model_name=_judge, device=DEVICE)
evaluator = Evaluator(judges=[judge], seed=SEED)
results_df, summary_df = evaluator.run(requests=requests, references=references,
                                       responses=responses, output_dir=RUN_DIR, track=None)
del judge, evaluator
print(f"scored {len(results_df)} items ({len(items)} orig + {len(items)} flip)")

In [ ]:
# --- read 1: paired accuracy delta, video-clustered CI ------------------------------
if SMOKE:
    print("SMOKE -- wiring only, no verdict")
else:
    orig = results_df[~results_df["qID"].str.endswith("__flip")].copy()
    flip = results_df[results_df["qID"].str.endswith("__flip")].copy()
    flip["qID"] = flip["qID"].str.removesuffix("__flip")
    assert len(orig) == len(flip) == len(items)

    print(f"accuracy, original frames : {orig['correctness'].mean():.4f}")
    print(f"accuracy, flipped frames  : {flip['correctness'].mean():.4f}")

    j = orig[["qID", "video", "primary", "correctness"]].rename(columns={"correctness": "correct_a"})
    j = j.merge(flip[["qID", "correctness"]].rename(columns={"correctness": "correct_b"}), on="qID")
    j["group"] = j["primary"].map(metrics._leaf_to_group)

    rows = [{"cell": "ALL", **metrics.paired_delta_ci(j, n_boot=N_BOOT, seed=SEED)}]
    for grp in sorted(j["group"].unique()):
        rows.append({"cell": grp, **metrics.paired_delta_ci(j[j["group"] == grp],
                                                             n_boot=N_BOOT, seed=SEED)})
    ci_df = pd.DataFrame(rows)
    ci_df["excludes_zero"] = (ci_df.ci_low > 0) | (ci_df.ci_high < 0)
    print("\nflip MINUS original, video-clustered CI:")
    print(ci_df.to_string(index=False))
    print("\n(RULES \u00a7S4: |\u0394| < 0.01 unreadable. n = 8 videos -- every CI here is wide by"
          " construction, same caveat rung 42's own held-out eval carries.)")

In [ ]:
# --- read 2: literal answer equivariance (judge-free) --------------------------------
# Only `object_center_quadrant` and `all_object_positions` carry a quadrant token in the
# ANSWER -- `fixed_quadrant_class`'s answer is a bare class name, so swapping left/right in
# it is a no-op by construction and this check does not apply there (its read is the paired
# accuracy delta above, scoped to that rule).
if SMOKE:
    print("SMOKE -- wiring only, no verdict")
else:
    quadrant_rules = {"object_center_quadrant", "all_object_positions"}
    eq_rows = []
    for it in items:
        qid = it.request.qID
        rule = audit_by_qid.loc[qid, "rule"]
        if rule not in quadrant_rules:
            continue
        p_orig = str(preds_by_qid.loc[qid, "prediction"])
        p_flip = str(preds_by_qid.loc[f"{qid}__flip", "prediction"])
        expected = flip_audit.swap_left_right_quadrants(p_orig)
        eq_rows.append({"qID": qid, "rule": rule, "orig_prediction": p_orig,
                        "flip_prediction": p_flip, "expected_if_equivariant": expected,
                        "literally_equivariant": expected.strip() == p_flip.strip()})
    eq_df = pd.DataFrame(eq_rows)
    rate = eq_df["literally_equivariant"].mean() if len(eq_df) else float("nan")
    print(f"literal answer equivariance, {len(eq_df)} rows (both quadrant-answer rules): "
          f"{rate:.1%}")
    print(eq_df.groupby("rule")["literally_equivariant"].mean())

In [ ]:
# --- persist --------------------------------------------------------------------
if not SMOKE:
    results_df.to_csv(EXP / "RESULTS_flip_pair_scored.csv", index=False)
    ci_df.to_csv(EXP / "RESULTS_flip_pair_paired_ci.csv", index=False)
    eq_df.to_csv(EXP / "RESULTS_flip_pair_equivariance.csv", index=False)
    (EXP / "RESULTS_flip_pair_summary.json").write_text(json.dumps({
        "checkpoint": "rung 42 ep4 (checkpoint-4848, submission 03)",
        "n_rows": len(items),
        "accuracy_original": float(orig["correctness"].mean()),
        "accuracy_flipped": float(flip["correctness"].mean()),
        "literal_equivariance_rate": float(rate) if rate == rate else None,
        "literal_equivariance_n": int(len(eq_df)),
    }, indent=2), encoding="utf-8")
    print("wrote RESULTS_flip_pair_scored.csv, RESULTS_flip_pair_paired_ci.csv, "
          "RESULTS_flip_pair_equivariance.csv, RESULTS_flip_pair_summary.json")
else:
    print("SMOKE -- nothing persisted")

In [ ]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) ----------
n_show = min(10, len(items))
for it in items[:n_show]:
    qid = it.request.qID
    row = audit_by_qid.loc[qid]
    print(f"[{row['rule']}] {qid}")
    print(f"  Q orig: {row['question']}")
    print(f"  Q flip: {row['flipped_question']}")
    print(f"  gold orig/flip: {row['answer']!r} / {row['flipped_answer']!r}")
    print(f"  pred orig/flip: {preds_by_qid.loc[qid, 'prediction']!r} / "
          f"{preds_by_qid.loc[f'{qid}__flip', 'prediction']!r}")
    print()